# GPT2-TTS - Colab training (L4)

Minimal **training-only** notebook. Loads the pre-tokenized `.pt` caches from Google
Drive (no re-upload, no FocalCodec, no raw audio needed) and trains GPT2-TTS on
**LJSpeech + pony + VCTK** together, saving `best`/`last` checkpoints and CSV logs
back to Drive.

The dataset / config / model / LightningModule cells are **exact copies** of
`lab4_gpt_tts.ipynb`.

**One-time setup:** upload the tokenized caches to Drive, mirroring the local layout:
```
MyDrive/kse-deep-audio/data/LJSpeech-1.1/tokenized/{train,val,test}.pt
MyDrive/kse-deep-audio/data/pony-speech/tokenized/{train,val,test}.pt
MyDrive/kse-deep-audio/data/vctk/tokenized/{train,val,test}.pt
```
Then Runtime -> Change runtime type -> **L4 GPU** and run top to bottom.

In [ ]:
!pip install -q lightning transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path


DRIVE = Path('/content/drive/MyDrive/kse-deep-audio')
DATA  = DRIVE / 'data'

CACHES = {
    'ljspeech': DATA / 'LJSpeech-1.1' / 'tokenized',
    'pony':     DATA / 'pony-speech'  / 'tokenized',
    'vctk':     DATA / 'vctk'         / 'tokenized',
}
CKPT_DIR = DRIVE / 'checkpoints' / 'gpt_tts_all'  
CKPT_DIR.mkdir(parents=True, exist_ok=True)

for k, v in CACHES.items():
    print(f'{k:>9}: exists={v.exists()}  ({v})')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from dataclasses import dataclass
from transformers import GPT2LMHeadModel
import lightning as L
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

In [ ]:
class TokenizedLJSpeech(Dataset):
  def __init__(self, cache_path, max_total_len=1024):
    items = torch.load(cache_path, weights_only=False)
    self.items = [it for it in items if it["n_text"] + it["n_audio"] + 2 <= max_total_len]

  def __len__(self):
    return len(self.items)

  def __getitem__(self, i):
    it = self.items[i]
    return {
      "file": it["file"],
      "speaker_id": 0,
      "text_ids": it["text_ids"].long(),
      "audio_ids": it["audio_ids"].long(),
    }

class TokenizedPony(Dataset):
  SPEAKERS = {
    "Twilight Sparkle": 1,
    "Applejack": 2,
    "Rainbow Dash": 3,
    "Rarity": 4,
    "Pinkie Pie": 5,
    "Spike": 6,
    "Fluttershy": 7,
    "Apple Bloom": 8,
    "Starlight": 9,
    "Scootaloo": 10,
  }

  def __init__(self, cache_path, max_total_len=1024, n_special=3):
    items = torch.load(cache_path, weights_only=False)
    self.items = [it for it in items if it["n_text"] + it["n_audio"] + n_special <= max_total_len]

  def __len__(self):
    return len(self.items)

  def __getitem__(self, i):
    it = self.items[i]
    return {
      "file": it["file"],
      "speaker_id": self.SPEAKERS[it["speaker"]],
      "text_ids": it["text_ids"].long(),
      "audio_ids": it["audio_ids"].long(),
    }
    
class TokenizedVCTK(Dataset):
  def __init__(self, cache_path, max_total_len=1024, n_special=3):
    items = torch.load(cache_path, weights_only=False)
    self.items = [it for it in items if it["n_text"]+it["n_audio"]+n_special <= max_total_len]
  def __len__(self): return len(self.items)
  def __getitem__(self, i):
    it = self.items[i]
    return {"file": it["file"], "speaker_id": it["speaker_id"],
            "text_ids": it["text_ids"].long(), "audio_ids": it["audio_ids"].long()}

def collate(batch):
  B = len(batch)
  text_lens = torch.tensor([b["text_ids"].numel() for b in batch], dtype=torch.long)
  audio_lens = torch.tensor([b["audio_ids"].numel() for b in batch], dtype=torch.long)
  text_ids = torch.zeros(B, int(text_lens.max()), dtype=torch.long)
  audio_ids = torch.zeros(B, int(audio_lens.max()), dtype=torch.long)
  speaker_ids = torch.tensor([b["speaker_id"] for b in batch], dtype=torch.long)

  for i, b in enumerate(batch):
    text_ids[i, : b["text_ids"].numel()] = b["text_ids"]
    audio_ids[i, : b["audio_ids"].numel()] = b["audio_ids"]

  return {
    "text_ids": text_ids,
    "text_lens": text_lens,
    "audio_ids": audio_ids,
    "audio_lens": audio_lens,
    "speaker_id": speaker_ids,
  }

In [ ]:
@dataclass
class GPT2TTSConfig:
  base_model: str = "gpt2"
  codebook_size: int = 8192
  n_special_tokens: int = 2  # BOS, EOS
  n_speakers: int = 119
  freezing_depth: int = 6

  @property
  def audio_vocab_size(self) -> int:
    return self.codebook_size + self.n_special_tokens

  @property
  def bos_id(self) -> int:
    return self.codebook_size

  @property
  def eos_id(self) -> int:
    return self.codebook_size + 1


In [ ]:
class GPT2TTS(nn.Module):
  def __init__(self, cfg: GPT2TTSConfig):
    super().__init__()
    self.cfg = cfg
    self.base = GPT2LMHeadModel.from_pretrained(cfg.base_model)
    H = self.base.config.n_embd
    V = cfg.audio_vocab_size

    self.audio_emb = nn.Embedding(V, H)
    self.audio_head = nn.Linear(H, V, bias=False)
    self.spk_emb = nn.Embedding(cfg.n_speakers, H)

    nn.init.normal_(self.spk_emb.weight, std=0.02)
    nn.init.normal_(self.audio_emb.weight, std=0.02)
    nn.init.normal_(self.audio_head.weight, std=0.02)

  def _build_inputs(self, text_ids, text_lens, audio_ids, audio_lens, speaker_id):
    """Pack a variable-length batch into a single (B, L, H) tensor with mask + labels.

    Inputs
    ------
    text_ids   : LongTensor[B, T_text_max]   padded GPT-2 BPE ids
    text_lens  : LongTensor[B]               actual text length per row
    audio_ids  : LongTensor[B, T_audio_max]  padded FocalCodec ids (all < codebook_size)
    audio_lens : LongTensor[B]               actual audio length per row

    Per-row layout (let tl = text_lens[i], al = audio_lens[i]):

      position :  0 .. tl-1   tl     tl+1 .. tl+al    tl+1+al    tl+2+al .. L-1
      content  :   text       BOS         audio          EOS          PAD

    Returns
    -------
    inputs : FloatTensor[B, L, H]
      Embeddings. Text goes through GPT-2's wte; audio/BOS/EOS through self.audio_emb.
    mask   : LongTensor[B, L]
      1 on real tokens, 0 on PAD. This is the `attention_mask` for GPT-2.
    labels : LongTensor[B, L]
      Target audio ids at positions that SHOULD be predicted (audio + EOS),
      and -100 everywhere else (text, BOS, PAD).
      Do NOT apply the causal shift here — `forward` does that.
    """

    # Gotchas
    # - L must be the padded max across the batch: max_i(tl + al + 2).
    # - Padded positions must be 0 in `mask` AND -100 in `labels` (or pass another "ignore" value to loss function)
    # - The BOS position is NOT a label (we don't train to predict BOS from text).
    # - The EOS position IS a label (we want the model to learn when to stop).
    # - text_emb and audio_emb have the same hidden size H — concatenate on time.

    inputs, mask, labels = None, None, None
    # ↓↓↓ your code here ↓↓↓
    assert text_lens.dim() == 1 and audio_lens.dim() == 1

    B = len(text_lens)
    L = (text_lens + audio_lens + 3).max().item()
    L = int(L)
    H = self.base.config.n_embd
    pad = self.base.transformer.wte.weight[self.base.config.eos_token_id]

    inputs = pad.expand(B, L, -1).clone()
    mask = torch.zeros(B, L, device=text_ids.device, dtype=torch.long)
    labels = torch.full((B, L), fill_value=-100, device=text_ids.device, dtype=torch.long)

    bos = self.audio_emb.weight[self.cfg.bos_id].unsqueeze(0)
    eos = self.audio_emb.weight[self.cfg.eos_id].unsqueeze(0)

    text_lens = text_lens.cpu().tolist()
    audio_lens = audio_lens.cpu().tolist()

    for i in range(B):
      audio_len = audio_lens[i]
      text_len = text_lens[i]
      spk = self.spk_emb(speaker_id[i]).unsqueeze(0)

      text = text_ids[i, :text_len]
      audio = audio_ids[i, :audio_len]

      text_embs = self.base.transformer.wte(text)
      audio_embs = self.audio_emb(audio)

      embs = torch.cat(
        [spk, text_embs, bos, audio_embs, eos], dim=0
      )

      inputs[i, :embs.shape[0]] = embs
      mask[i, :embs.shape[0]] = 1
      labels[i, text_len + 2:text_len + 2 + audio_len] = audio
      labels[i, text_len + 2 + audio_len] = self.cfg.eos_id
    # ↑↑↑ your code here ↑↑↑

    return inputs, mask, labels

  def forward(self, text_ids, text_lens, audio_ids, audio_lens, speaker_id):
    inputs, mask, labels = self._build_inputs(text_ids, text_lens, audio_ids, audio_lens, speaker_id)
    h = self.base.transformer(
      inputs_embeds=inputs, attention_mask=mask, return_dict=True
    ).last_hidden_state
    logits = self.audio_head(h)  # (B, L, V)

    # Compute the causal-LM cross-entropy loss.
    # In a causal LM the hidden state at position t predicts the token at position t+1.
    # Use ignore_index=-100 so masked positions don't contribute. Cast logits to float32
    # before the softmax for numerical stability under bf16/fp16.
    # Expected: a scalar tensor `loss`.

    loss = None
    # ↓↓↓ your code here ↓↓↓
    V = self.cfg.audio_vocab_size

    loss = F.cross_entropy(
      logits[:, :-1, :].float().reshape(-1, V),
      labels[:, 1:].reshape(-1),
      ignore_index=-100,
    )

    # ↑↑↑ your code here ↑↑↑

    return {"loss": loss, "logits": logits}

  @torch.no_grad()
  def generate_audio(self, text_ids, speaker_id, max_new_tokens=400, temperature=0.5, top_k=10):
    """Autoregressively sample audio tokens conditioned on text.

    Runs [text][BOS] through the transformer once to prime the KV cache,
    leaving `h` as the hidden state after BOS. You write the sampling loop.
    """
    self.eval()
    device = text_ids.device
    text_ids = text_ids.view(-1).long().to(device)
    spk = self.spk_emb.weight[speaker_id].view(1, 1, -1)
    text_emb = self.base.transformer.wte(text_ids).unsqueeze(0)                        # (1, T, H)
    bos = self.audio_emb.weight[self.cfg.bos_id].view(1, 1, -1)            # (1, 1, H)
    prefix = torch.cat([spk, text_emb, bos], dim=1)                             # (1, T+1, H)
    mask = torch.ones(1, prefix.size(1), device=device, dtype=torch.long)

    out = self.base.transformer(
      inputs_embeds=prefix, attention_mask=mask,
      use_cache=True, return_dict=True,
    )
    past = out.past_key_values
    h = out.last_hidden_state[:, -1, :]   # (1, H) — hidden state after BOS

    # Autoregressive sampling loop. Starting from `h` (hidden state after BOS), repeatedly:
    # project to audio-vocab logits, apply temperature + top-k, sample a token, handle
    # stopping, and feed the token back through
    # the transformer using `past`/`mask` to refresh `h`. Stop at EOS or max_new_tokens.
    # Return the collected audio ids as a LongTensor on `device`.

    generated = []
    # ↓↓↓ your code here ↓↓↓
    for _ in range(max_new_tokens):
      logits = self.audio_head(h) / temperature
      logits[:, self.cfg.bos_id] = -float('inf')

      if top_k is not None and top_k > 0:
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits[logits < v[:, [-1]]] = -float('Inf')

      probs = F.softmax(logits, dim=-1)
      idx_next = torch.multinomial(probs, num_samples=1)

      if idx_next.item() == self.cfg.eos_id:
        break

      generated.append(idx_next)

      next_emb = self.audio_emb(idx_next)

      mask = torch.ones(1, prefix.size(1) + len(generated), device=device, dtype=torch.long)

      out = self.base.transformer(
        inputs_embeds=next_emb, attention_mask=mask,
        use_cache=True, return_dict=True,
        past_key_values=past,
      )

      past = out.past_key_values
      h = out.last_hidden_state[:, -1, :]
    # ↑↑↑ your code here ↑↑↑

    return torch.tensor(generated, dtype=torch.long, device=device)


In [ ]:
class SpeechGPT(L.LightningModule):
  def __init__(self, model):
    super().__init__()
    self.model = model

  def training_step(self, batch, _):
    loss = self.model(**batch)["loss"]
    self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def validation_step(self, batch, _):
    out = self.model(**batch)
    self.log("val_loss", out["loss"], on_step=False, on_epoch=True, prog_bar=True)

  def on_validation_epoch_end(self):
    g = torch.Generator().manual_seed(0)
    tr, sr = [], []
    for k, b in enumerate(self.trainer.val_dataloaders):
      if k >= 8: break
      b = {k2: v.to(self.device) for k2, v in b.items()}
      n = b['text_ids'].size(0)
      perm = torch.randperm(n, generator=g)
      if torch.equal(perm, torch.arange(n)):
        perm = perm.roll(1)
      perm = perm.to(self.device)
      with torch.no_grad():
        clean = self.model(**b)['loss']
        shuf_t = self.model(**{**b, 'text_ids': b['text_ids'][perm],
                                    'text_lens': b['text_lens'][perm]})['loss']
        n_spk = self.model.cfg.n_speakers
        alt_spk = (b['speaker_id'] + 1) % n_spk
        shuf_s = self.model(**{**b, 'speaker_id': alt_spk})['loss']
      tr.append((shuf_t - clean).item())
      sr.append((shuf_s - clean).item())
    self.log('text_reliance', sum(tr)/len(tr), prog_bar=True)
    self.log('spk_reliance', sum(sr)/len(sr), prog_bar=True)

  def configure_optimizers(self):
    decay, no_decay, new = [], [], []

    for n, p in self.model.named_parameters():
      if not p.requires_grad:
        continue
      if n.startswith(('audio_', 'spk_')):
        new.append(p)
      elif p.dim() < 2:
        no_decay.append(p)
      else:
        decay.append(p)

    opt = torch.optim.AdamW([
      {'params': decay, 'lr': 2e-5, 'weight_decay': 0.01},
      {'params': no_decay, 'lr': 2e-5, 'weight_decay': 0.0},
      {'params': new, 'lr': 2e-4, 'weight_decay': 0.01},
    ])

    # sched = torch.optim.lr_scheduler.OneCycleLR(
    #   opt, max_lr=[1e-4, 1e-4, 1e-3],
    #   total_steps=self.trainer.estimated_stepping_batches,
    #   pct_start=0.05, anneal_strategy='cos',
    # )

    return opt#, 'lr_scheduler': {'scheduler': sched, 'interval': 'step'}}


In [ ]:
BATCH   = 32
WORKERS = 4   

def concat(split):
  return ConcatDataset([
    TokenizedLJSpeech(CACHES["ljspeech"] / f"{split}.pt"),
    TokenizedPony(CACHES["pony"] / f"{split}.pt"),
    TokenizedVCTK(CACHES["vctk"] / f"{split}.pt"),
  ])

train_ds, val_ds = concat("train"), concat("val")
print(f"train={len(train_ds)}  val={len(val_ds)}")

train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, collate_fn=collate,
                      num_workers=WORKERS, pin_memory=True, persistent_workers=True)
val_dl   = DataLoader(val_ds, batch_size=BATCH, collate_fn=collate,
                      num_workers=WORKERS, pin_memory=True, persistent_workers=True)

In [ ]:
cfg = GPT2TTSConfig()
cfg = GPT2TTSConfig(base_model=str(DRIVE / "gpt2"))

ckpt = ModelCheckpoint(dirpath=str(CKPT_DIR), monitor="val_loss", mode="min",
                       save_top_k=1, save_last=True, filename="best")
early = EarlyStopping(monitor="val_loss", mode="min", patience=10, min_delta=1e-3)
logger = CSVLogger(str(DRIVE / "logs"), name="gpt_tts_all")

trainer = L.Trainer(
  accelerator="gpu", devices=1, precision="bf16-mixed",
  max_epochs=30, gradient_clip_val=1.0,
  callbacks=[ckpt, early], logger=logger,
)

torch.set_float32_matmul_precision("high")
resume = CKPT_DIR / "last.ckpt"                    
trainer.fit(model, train_dl, val_dl,
            ckpt_path=str(resume) if resume.exists() else None)

print("best  :", ckpt.best_model_path)
print("last  :", CKPT_DIR / "last.ckpt")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

LOG_ROOT = DRIVE / "logs" / "gpt_tts_all"
versions = sorted(LOG_ROOT.glob("version_*"), key=lambda p: int(p.name.split("_")[1]))
csv = versions[-1] / "metrics.csv"
print("reading", csv)

metrics = pd.read_csv(csv)
m = metrics.groupby("epoch").last()   

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

for col in ["train_loss", "val_loss"]:
  if col in m:
    ax[0].plot(m.index, m[col], marker="o", label=col)
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].set_title("Loss")
ax[0].legend(); ax[0].grid(alpha=0.3)

for col in ["text_reliance", "spk_reliance"]:
  if col in m:
    ax[1].plot(m.index, m[col], marker="o", label=col)
ax[1].axhline(0, color="k", lw=0.8)
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("delta loss (shuffled - clean)")
ax[1].set_title("Reliance (higher = uses that input more)")
ax[1].legend(); ax[1].grid(alpha=0.3)

fig.tight_layout(); plt.show()